In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
import cv2  
import matplotlib.pyplot as plt
# Dataset paths
dataset_path = "/kaggle/input/moddelite/ModdeDataset"
jpeg_dataset_path = "/kaggle/working/MODDE_JPEG"

# Check if dataset path exists
if os.path.exists(dataset_path):
    print("Dataset exists.")
else:
    print("Paths aren't aligned properly. Please check the dataset path.")

# Function to convert images to JPEG format
def convert_images_to_jpeg(src_dir, dest_dir):
    # Create the destination directory if it doesn't exist
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)

    # Traverse through the source directory
    for root, _, files in os.walk(src_dir):
        for file in files:
            src_file_path = os.path.join(root, file)  # Source file path
            rel_path = os.path.relpath(root, src_dir)  # Relative path
            dest_dir_path = os.path.join(dest_dir, rel_path)  # Destination directory
            dest_file_path = os.path.join(dest_dir_path, file)  # Destination file path

            # Ensure the destination directory exists
            if not os.path.exists(dest_dir_path):
                os.makedirs(dest_dir_path)

            # Skip files that already exist
            if os.path.exists(dest_file_path):
                continue

            try:
                # Convert and save as JPEG
                with Image.open(src_file_path) as img:
                    if img.mode != 'RGB':  # Convert to RGB if needed
                        img = img.convert('RGB')
                    dest_file_path = os.path.splitext(dest_file_path)[0] + '.jpg'
                    img.save(dest_file_path, 'JPEG')
            except Exception as e:
                print(f"Failed to process {src_file_path}: {e}")

# Convert images to JPEG format
convert_images_to_jpeg(dataset_path, jpeg_dataset_path)

# Load the dataset
data = tf.keras.utils.image_dataset_from_directory(
    jpeg_dataset_path,
    labels='inferred',  # Automatically infer labels based on folder names
    label_mode='categorical',  # 'int', 'categorical', or None
    batch_size=32,  # Number of images per batch
    image_size=(256, 256),  # Resize images to a uniform size
    shuffle=True,  # Shuffle the dataset
    seed=123  # Seed for reproducibility
)

print(data.class_names)  # Check if the class names match your folder structure

# Optional: Print a summary of the dataset
for images, labels in data.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Label batch shape: {labels.shape}")


Dataset exists.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(14, 10))  # Adjust the figure size for better spacing
shown_classes = set()  # Track displayed classes
grid_size = (3, 4)  # Grid size: Adjust according to the number of classes

# Iterate through the dataset to display one sample per class
for images, labels in data:
    for i in range(len(images)):
        label_index = np.argmax(labels[i].numpy())  # Get the class index
        if label_index not in shown_classes:
            ax = plt.subplot(grid_size[0], grid_size[1], len(shown_classes) + 1)  # Place in grid
            ax.imshow(images[i].numpy().astype(int))
            ax.set_title(
                data.class_names[label_index], 
                fontsize=12, 
                weight="bold", 
                color="#333333"  # Modern gray tone for title
            )
            ax.axis("off")  # Remove axes for a clean look
            shown_classes.add(label_index)
        if len(shown_classes) == len(data.class_names):  # Break if all classes are displayed
            break
    if len(shown_classes) == len(data.class_names):  # Break outer loop if done
        break

# Add a central title for the whole grid
plt.suptitle(
    "Sample Images from Each Class",
    fontsize=16,
    weight="bold",
    color="#222222",
    y=0.92
)

# Adjust layout for better spacing
plt.tight_layout(pad=2.0, rect=[0, 0, 1, 0.92])
plt.show()


In [ ]:
data=data.map(lambda x,y:(x/255,y)) #transformation for this pipline
data.as_numpy_iterator().next()[0].max()

In [ ]:
# Get the total number of batches in the dataset
total_batches = len(data)

# Calculate the number of batches for training, validation, and test
train_size = int(total_batches * 0.8)
val_size = int(total_batches * 0.1)
test_size = total_batches - train_size - val_size  # Remaining for test

# Split the dataset into train, validation, and test sets
train = data.take(train_size)
val = data.skip(train_size).take(val_size)
test = data.skip(train_size + val_size).take(test_size)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential()

# Convolutional Layer and MaxPooling Layer
model.add(Conv2D(16, (3, 3), strides=1, activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D())

model.add(Conv2D(32, (3, 3), strides=1, activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(16, (3, 3), strides=1, activation='relu'))
model.add(MaxPooling2D())

# Flatten and Dense Layers
model.add(Flatten())
model.add(Dense(256, activation="relu"))

# Adjusted final Dense layer for multiclass classification
model.add(Dense(12, activation="softmax"))  # Replace 12 with the number of your classes

# Compile the model
model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Summary of the model
model.summary()


In [ ]:
logdir="logs"
tensorboard_callback=tf.keras.callbacks.TensorBoard(log_dir=logdir)


In [ ]:
# Training the model
history = model.fit(
    train,  # Training dataset
    epochs=20,  # Number of epochs
    validation_data=val,  # Validation dataset
    callbacks=[tensorboard_callback]  # TensorBoard callback
)

# Calculate total accuracy and loss
mod_accuracy = history.history['accuracy'][-1]  # Accuracy from the last epoch
mod_loss = history.history['loss'][-1]  # Loss from the last epoch

print(f"Model Total Accuracy: {mod_accuracy * 100:.2f}%")
print(f"Model Total Loss: {mod_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Define total accuracy and loss
mod_accuracy = 0.9878  # Replace with actual value
mod_loss = 0.0393  # Replace with actual value

# Simulate history data for the plot
history = {
    "loss": [0.0472, 0.0297, 0.0274, 0.0324, 0.0164, 0.0154, 0.0466, 0.1340, 0.0483, 0.0300],
    "val_loss": [0.8465, 0.9733, 0.8848, 0.9410, 0.8992, 0.8569, 1.0634, 0.9834, 0.9806, 0.9311],
    "accuracy": [0.9873, 0.9915, 0.9922, 0.9895, 0.9944, 0.9946, 0.9872, 0.9591, 0.9850, 0.9897],
    "val_accuracy": [0.8551, 0.8629, 0.8651, 0.8572, 0.8558, 0.8629, 0.8217, 0.8459, 0.8473, 0.8530],
}

# Set up the dark theme
plt.style.use("dark_background")
plt.figure(figsize=(10, 6))

# Use a modern font
font_path = fm.findfont(fm.FontProperties(family="Arial"))
font_properties = fm.FontProperties(fname=font_path)

# Plot the data with adjusted line width
plt.plot(history["loss"], color="#1f77b4", linewidth=2, label="Loss")
#plt.plot(history["val_loss"], color="#ff7f0e", linewidth=2, label="Validation Loss")
plt.plot(history["accuracy"], color="#d62728", linewidth=2, label="Accuracy")
plt.plot(history["val_accuracy"], color="#2ca02c", linewidth=2, label="Validation Accuracy")

# Add title and labels
plt.title("Model Training History", fontsize=16, fontproperties=font_properties, pad=20)
plt.xlabel("Epochs", fontsize=14, fontproperties=font_properties, labelpad=10)
plt.ylabel("Metric Values", fontsize=14, fontproperties=font_properties, labelpad=10)

# Add legend outside the plot
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=12, frameon=False)

# Add total accuracy and loss as annotations
text = f"Model Total Accuracy: {mod_accuracy * 100:.2f}%\nModel Total Loss: {mod_loss:.4f}"
plt.gcf().text(0.95, 0.5, text, fontsize=12, fontproperties=font_properties, color="white", ha="right", va="center")

# Adjust layout
plt.tight_layout(rect=[0, 0, 0.9, 1])

# Show the plot
plt.show()


In [ ]:
from tensorflow.keras.metrics import Precision, Recall, CategoricalAccuracy

# Define metrics for multiclass classification
pre = Precision()  # Precision for multiclass classification
re = Recall()      # Recall for multiclass classification
acc = CategoricalAccuracy()  # Use CategoricalAccuracy for one-hot encoded labels

In [ ]:
len(test)